# Day 3：手写 Agent 实现 — 从 Tool Registry 到 ReAct Loop> **目标**：从零实现一个最小可运行的 Agent 系统，理解 Tool 注册、JSON Schema 参数校验、Action 解析、Observation 回填和多步循环的核心逻辑；实现错误处理与多轮 Memory 管理；先用可控的 mock planner 把系统骨架写通，再为后续接入真实模型做好准备。---## 总体流程```textPart 1: 定义 Tool 抽象与注册表（含 JSON Schema）Part 2: 准备可调用工具Part 3: JSON Schema 参数校验Part 4: 错误处理与 ToolResult 规范化Part 5: 实现最小 ReAct AgentPart 6: 多轮 Memory 管理Part 7: 带 Memory 的多轮 AgentPart 8: 完整测试场景Part 9: 局限分析与面试要点```

In [ ]:
from __future__ import annotationsfrom dataclasses import dataclass, fieldfrom typing import Any, Callable, Dict, List, Optionalimport jsonimport mathimport traceback

## Part 1：定义 Tool 与 ToolRegistry（含 JSON Schema）Agent 需要先知道自己"可以做什么"。与 Day2 介绍的 Function Calling 对齐，每个工具除了 `name` 和 `func`，还需要用 JSON Schema 描述参数结构，这样系统才能对模型输出做校验。

In [ ]:
@dataclassclass Tool:    """一个可调用工具的完整描述。"""    name: str    description: str    func: Callable[..., Any]    parameters: Dict[str, Any] = field(default_factory=dict)    def run(self, **kwargs) -> Any:        return self.func(**kwargs)    def to_schema(self) -> Dict[str, Any]:        """生成 Function Calling 风格的 JSON Schema 描述。"""        return {            "name": self.name,            "description": self.description,            "parameters": self.parameters,        }class ToolRegistry:    """工具注册表：统一管理所有可用工具。"""    def __init__(self):        self.tools: Dict[str, Tool] = {}    def register(self, tool: Tool):        if tool.name in self.tools:            raise ValueError(f"工具 {tool.name} 已存在")        self.tools[tool.name] = tool    def get(self, name: str) -> Tool:        if name not in self.tools:            raise KeyError(f"未知工具: {name}")        return self.tools[name]    def list_names(self) -> List[str]:        return list(self.tools.keys())    def describe(self) -> List[Dict[str, Any]]:        """返回所有工具的 JSON Schema 列表，供 LLM 使用。"""        return [tool.to_schema() for tool in self.tools.values()]    def describe_text(self) -> str:        """返回人类可读的工具描述（用于构造 prompt）。"""        lines = []        for tool in self.tools.values():            params = tool.parameters.get("properties", {})            param_str = ", ".join(                f'{k}: {v.get("type", "?")}' for k, v in params.items()            )            lines.append(f"- {tool.name}({param_str}): {tool.description}")        return "\n".join(lines)

In [ ]:
demo_tool = Tool(    name="demo",    description="演示工具",    func=lambda x: x * 2,    parameters={        "type": "object",        "properties": {            "x": {"type": "integer", "description": "输入数字"}        },        "required": ["x"],    },)print("Schema:", json.dumps(demo_tool.to_schema(), ensure_ascii=False, indent=2))print("Result:", demo_tool.run(x=21))

## Part 2：准备可调用工具为了突出 Agent 机制本身，这里准备四个零依赖工具：课程资料检索、计算器、天气查询和时间查询。每个工具都带完整的 JSON Schema。

In [ ]:
COURSE_KB = {    "lora": {        "content": "LoRA 是一种参数高效微调方法，用低秩矩阵 BA 近似权重更新 DeltaW。",        "source": "Week6_Day2",    },    "qlora": {        "content": "QLoRA 在 LoRA 基础上引入 NF4 量化和双重量化，大幅降低微调显存。",        "source": "Week6_Day4",    },    "rag": {        "content": "RAG 是检索增强生成，通过外部知识库补充模型参数外的知识。",        "source": "Week8_Day4",    },    "react": {        "content": "ReAct 让模型在 Thought、Action、Observation 之间循环，适合工具调用场景。",        "source": "Week8_Day2",    },    "transformer": {        "content": "Transformer 使用自注意力机制处理序列，是现代大语言模型的基础架构。",        "source": "Week2_Day1",    },    "infonce": {        "content": "InfoNCE 是对比学习的核心损失函数，通过 softmax 形式区分正负样本。",        "source": "Week8_Day5",    },}def search_course_notes(query: str, top_k: int = 2) -> List[Dict[str, str]]:    """基于关键词匹配检索课程资料，返回带来源的结果。"""    query_lower = query.lower()    hits = []    for key, entry in COURSE_KB.items():        score = sum(            word in entry["content"].lower() or word in key            for word in query_lower.split()        )        if score > 0:            hits.append((score, key, entry))    hits.sort(reverse=True)    return [        {"key": key, "content": entry["content"], "source": entry["source"]}        for _, key, entry in hits[:top_k]    ]def calculator(expression: str) -> float:    """执行简单数学表达式，支持 +, -, *, /, sqrt, abs, round。"""    allowed_names = {"abs": abs, "round": round, "sqrt": math.sqrt, "pow": pow}    return eval(expression, {"__builtins__": {}}, allowed_names)def weather_api(city: str) -> Dict[str, Any]:    """查询指定城市的天气信息。"""    fake_weather = {        "北京": {"weather": "多云", "temp": 18, "aqi": "良", "wind": "北风2级"},        "上海": {"weather": "小雨", "temp": 22, "aqi": "中", "wind": "东风3级"},        "深圳": {"weather": "晴", "temp": 28, "aqi": "优", "wind": "南风1级"},    }    result = fake_weather.get(city)    if result is None:        raise ValueError(f"不支持的城市: {city}")    return resultdef get_current_time() -> str:    """获取当前模拟时间。"""    return "2026-03-06 14:30:00 (CST)"

In [ ]:
registry = ToolRegistry()registry.register(Tool(    name="search_course_notes",    description="检索课程资料，返回相关知识点和来源",    func=search_course_notes,    parameters={        "type": "object",        "properties": {            "query": {"type": "string", "description": "检索关键词"},            "top_k": {"type": "integer", "description": "返回结果数量"},        },        "required": ["query"],    },))registry.register(Tool(    name="calculator",    description="执行数学表达式计算",    func=calculator,    parameters={        "type": "object",        "properties": {            "expression": {"type": "string", "description": "数学表达式"},        },        "required": ["expression"],    },))registry.register(Tool(    name="weather_api",    description="查询指定城市的天气",    func=weather_api,    parameters={        "type": "object",        "properties": {            "city": {"type": "string", "description": "城市名称"},        },        "required": ["city"],    },))registry.register(Tool(    name="get_current_time",    description="获取当前时间",    func=get_current_time,    parameters={"type": "object", "properties": {}},))print("已注册工具:", registry.list_names())print()print("工具描述（prompt 格式）:")print(registry.describe_text())print()print("JSON Schema 格式:")print(json.dumps(registry.describe(), ensure_ascii=False, indent=2))

## Part 3：JSON Schema 参数校验真实 Agent 系统中，模型输出的工具参数不一定合法。参数校验是工具执行前的必要防线：1. 必填字段是否存在2. 参数类型是否正确3. 是否有未知参数这里实现一个轻量版 JSON Schema 校验器（不依赖第三方库）。

In [ ]:
TYPE_MAP = {    "string": str,    "integer": int,    "number": (int, float),    "boolean": bool,    "array": list,    "object": dict,}def validate_tool_args(schema: Dict[str, Any], arguments: Dict[str, Any]) -> tuple:    """    根据 JSON Schema 校验工具参数。    Returns:        (is_valid: bool, error_message: str)    """    properties = schema.get("properties", {})    required = schema.get("required", [])    for req_field in required:        if req_field not in arguments:            return False, f"缺少必填参数: {req_field}"    for arg_name, arg_value in arguments.items():        if arg_name not in properties:            return False, f"未知参数: {arg_name}"        expected_type = properties[arg_name].get("type")        if expected_type and expected_type in TYPE_MAP:            if not isinstance(arg_value, TYPE_MAP[expected_type]):                return False, (                    f"参数 {arg_name} 类型错误: "                    f"期望 {expected_type}, 实际 {type(arg_value).__name__}"                )    return True, ""

In [ ]:
calc_schema = registry.get("calculator").parameterstests = [    ({"expression": "1+1"}, True, "合法参数"),    ({}, False, "缺少必填参数"),    ({"expression": 123}, False, "类型错误"),    ({"expression": "1+1", "unknown": True}, False, "未知参数"),]for args, expected_valid, desc in tests:    valid, msg = validate_tool_args(calc_schema, args)    status = "✅" if valid == expected_valid else "❌"    print(f"{status} {desc}: valid={valid}, msg='{msg}'")    assert valid == expected_validprint("\n✅ 所有校验测试通过")

## Part 4：错误处理与 ToolResult 规范化工具执行可能失败（参数非法、运行时异常、超时），Agent 需要稳定消费工具返回值。因此用 `ToolResult` 统一封装，确保无论成功还是失败，结构都可预测。

In [ ]:
@dataclassclass ToolResult:    """统一的工具执行结果。"""    status: str       # "ok" | "error"    result: Any       # 成功时的返回值    error: Optional[str] = None  # 失败时的错误信息    tool_name: str = ""    def to_observation(self) -> str:        """转为可读的 observation 字符串。"""        if self.status == "ok":            return json.dumps(self.result, ensure_ascii=False, default=str)        return f"[工具错误] {self.error}"def safe_execute_tool(    registry: ToolRegistry,    tool_name: str,    arguments: Dict[str, Any],) -> ToolResult:    """安全执行工具：校验参数 -> 运行函数 -> 捕获异常。"""    try:        tool = registry.get(tool_name)    except KeyError as e:        return ToolResult(status="error", result=None, error=str(e), tool_name=tool_name)    valid, msg = validate_tool_args(tool.parameters, arguments)    if not valid:        return ToolResult(status="error", result=None, error=msg, tool_name=tool_name)    try:        result = tool.run(**arguments)        return ToolResult(status="ok", result=result, tool_name=tool_name)    except Exception as e:        return ToolResult(            status="error", result=None,            error=f"{type(e).__name__}: {e}", tool_name=tool_name,        )

In [ ]:
# 1. 正常执行r1 = safe_execute_tool(registry, "calculator", {"expression": "23 * 17 + 5"})print(f"正常执行:  status={r1.status}, result={r1.result}")assert r1.status == "ok" and r1.result == 396# 2. 工具不存在r2 = safe_execute_tool(registry, "nonexistent", {})print(f"工具不存在: status={r2.status}, error={r2.error}")assert r2.status == "error"# 3. 参数缺失r3 = safe_execute_tool(registry, "calculator", {})print(f"参数缺失:  status={r3.status}, error={r3.error}")assert r3.status == "error"# 4. 运行时异常（除以零）r4 = safe_execute_tool(registry, "calculator", {"expression": "1/0"})print(f"运行异常:  status={r4.status}, error={r4.error}")assert r4.status == "error"# 5. 工具内部抛异常（不支持的城市）r5 = safe_execute_tool(registry, "weather_api", {"city": "火星"})print(f"业务异常:  status={r5.status}, error={r5.error}")assert r5.status == "error"print("\n✅ 所有安全执行测试通过")

## Part 5：实现最小 ReAct Agent真实系统里 planner 由 LLM 实现。为了保证 notebook 无外部依赖可运行，先用规则化 `mock_planner` 驱动 Agent Loop。Agent 的核心循环是：```textwhile step < max_steps:    decision = planner(query, scratchpad)    if decision is final_answer:        return answer    else:        observation = safe_execute_tool(decision)        scratchpad.append(thought, action, observation)```

In [ ]:
def mock_planner(    user_query: str,    scratchpad: List[Dict[str, Any]],    tools: List[str],) -> Dict[str, Any]:    """    规则化 planner，模拟 LLM 的决策过程。    真实系统中这里会调用 LLM，输入包括 user_query、scratchpad 和    tools 描述，输出是 tool_call 或 final_answer。    """    if scratchpad:        last = scratchpad[-1]        obs = last["observation"]        if last.get("tool_result_status") == "error":            return {                "type": "final_answer",                "content": f"工具调用失败了：{obs}。我直接根据已知信息回答。",            }        return {            "type": "final_answer",            "content": f"基于工具返回的信息，我的结论是：{obs}",        }    query_lower = user_query.lower()    if "天气" in user_query:        for city in ["北京", "上海", "深圳"]:            if city in user_query:                return {                    "type": "tool_call",                    "name": "weather_api",                    "arguments": {"city": city},                    "thought": f"用户在问{city}的天气，我需要先查询天气信息。",                }        return {            "type": "tool_call",            "name": "weather_api",            "arguments": {"city": "北京"},            "thought": "用户问天气但没指定城市，默认查北京。",        }    if "时间" in user_query or "几点" in user_query:        return {            "type": "tool_call",            "name": "get_current_time",            "arguments": {},            "thought": "用户在问时间，调用时间工具。",        }    if any(kw in query_lower for kw in ["lora", "rag", "react", "transformer", "infonce", "qlora"]):        return {            "type": "tool_call",            "name": "search_course_notes",            "arguments": {"query": user_query, "top_k": 2},            "thought": "这个问题涉及课程知识点，先检索资料。",        }    if any(op in user_query for op in ["+", "-", "*", "/", "计算", "等于"]):        expr = user_query.replace("计算", "").replace("等于多少", "").replace("等于", "").strip()        return {            "type": "tool_call",            "name": "calculator",            "arguments": {"expression": expr},            "thought": "用户需要计算，调用计算器避免手算错误。",        }    return {        "type": "final_answer",        "content": "这个问题不需要外部工具，我可以直接回答。",    }

In [ ]:
class SimpleReActAgent:    """    最小 ReAct Agent：Thought -> Action -> Observation 循环。    面试高频手写：这个类是 Agent 系统的核心骨架。    """    def __init__(        self,        planner: Callable,        registry: ToolRegistry,        max_steps: int = 5,        max_tool_calls: int = 3,        verbose: bool = True,    ):        self.planner = planner        self.registry = registry        self.max_steps = max_steps        self.max_tool_calls = max_tool_calls        self.verbose = verbose    def run(self, user_query: str) -> Dict[str, Any]:        scratchpad: List[Dict[str, Any]] = []        tool_call_count = 0        for step in range(self.max_steps):            decision = self.planner(                user_query, scratchpad, self.registry.list_names()            )            if decision["type"] == "final_answer":                if self.verbose:                    print(f"  [Final] {decision['content']}")                return {                    "query": user_query,                    "steps": scratchpad,                    "final_answer": decision["content"],                    "tool_calls": tool_call_count,                }            if tool_call_count >= self.max_tool_calls:                if self.verbose:                    print(f"  [Limit] 达到工具调用上限 ({self.max_tool_calls})")                return {                    "query": user_query,                    "steps": scratchpad,                    "final_answer": "工具调用次数已达上限，基于已有信息回答。",                    "tool_calls": tool_call_count,                }            tool_name = decision["name"]            arguments = decision["arguments"]            thought = decision.get("thought", "")            if self.verbose:                print(f"  [Step {step+1}] Thought: {thought}")                print(f"           Action: {tool_name}({arguments})")            tool_result = safe_execute_tool(self.registry, tool_name, arguments)            tool_call_count += 1            if self.verbose:                print(f"           Observation: {tool_result.to_observation()}")            scratchpad.append({                "thought": thought,                "action": tool_name,                "arguments": arguments,                "observation": tool_result.to_observation(),                "tool_result_status": tool_result.status,            })        return {            "query": user_query,            "steps": scratchpad,            "final_answer": "达到最大步骤数，任务提前终止。",            "tool_calls": tool_call_count,        }agent = SimpleReActAgent(mock_planner, registry)

In [ ]:
examples = [    "请解释 LoRA 和 RAG 的区别",    "北京今天天气适合跑步吗？",    "计算 23 * 17 + 5",    "现在几点了？",    "你好，今天过得怎么样？",]for query in examples:    print("=" * 70)    print(f"Query: {query}")    result = agent.run(query)    print(f"Tool calls: {result['tool_calls']}\n")

### 5.1 验证错误恢复能力Agent 必须能处理工具执行失败的场景。

In [ ]:
def planner_bad_city(query, scratchpad, tools):    if scratchpad:        last = scratchpad[-1]        if last["tool_result_status"] == "error":            return {                "type": "final_answer",                "content": "抱歉，无法查询该城市天气。目前只支持北京、上海、深圳。",            }        return {            "type": "final_answer",            "content": f"天气信息：{last['observation']}",        }    return {        "type": "tool_call",        "name": "weather_api",        "arguments": {"city": "广州"},        "thought": "查询广州天气",    }error_agent = SimpleReActAgent(planner_bad_city, registry)print("Query: 广州天气如何？")result = error_agent.run("广州天气如何？")print(f"\n最终答案: {result['final_answer']}")assert result["steps"][0]["tool_result_status"] == "error"print("✅ Agent 正确处理了工具执行失败")

### 5.2 验证工具调用次数限制

In [ ]:
def planner_always_call(query, scratchpad, tools):    """模拟一个永远调用工具的 planner，测试 max_tool_calls 限制。"""    return {        "type": "tool_call",        "name": "get_current_time",        "arguments": {},        "thought": f"第 {len(scratchpad)+1} 次调用...",    }limit_agent = SimpleReActAgent(    planner_always_call, registry, max_steps=10, max_tool_calls=3)print("测试工具调用次数限制 (max_tool_calls=3):")result = limit_agent.run("无限查时间")print(f"\n实际工具调用次数: {result['tool_calls']}")assert result["tool_calls"] == 3print("✅ max_tool_calls 限制生效")

## Part 6：多轮 Memory 管理单轮 Agent 结束后，scratchpad 就丢了。多轮对话要求 Agent 记住之前聊过什么。这里实现 `AgentMemory`，连接 Day1 的 token 预算概念：- 保存用户消息、助手回复、工具调用记录- 当 history 过长时，自动做滑窗截断- 提供格式化的上下文输出供 planner 使用

In [ ]:
def estimate_tokens(text: str) -> int:    """粗略估算 token 数（中文约 1 字 = 1.5 token，英文约 1 词 = 1 token）。"""    chinese_chars = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')    other_chars = len(text) - chinese_chars    return int(chinese_chars * 1.5 + other_chars * 0.3)@dataclassclass MemoryEntry:    role: str          # "user" | "assistant" | "tool_call"    content: str    metadata: Dict[str, Any] = field(default_factory=dict)    @property    def token_estimate(self) -> int:        return estimate_tokens(self.content)class AgentMemory:    """    Agent 的多轮记忆管理器。    短期记忆：当前和最近几轮的完整对话 + 工具调用    截断策略：当总 token 超过预算时，从最早的记录开始裁剪    """    def __init__(self, max_tokens: int = 2000):        self.entries: List[MemoryEntry] = []        self.max_tokens = max_tokens    def add(self, role: str, content: str, **metadata):        self.entries.append(MemoryEntry(role=role, content=content, metadata=metadata))        self._truncate_if_needed()    def add_user_message(self, content: str):        self.add("user", content)    def add_assistant_message(self, content: str):        self.add("assistant", content)    def add_tool_call(self, tool_name: str, arguments: Dict, result: ToolResult):        self.add(            "tool_call",            f"{tool_name}({json.dumps(arguments, ensure_ascii=False)}) -> {result.to_observation()}",            tool_name=tool_name,            status=result.status,        )    def get_context(self) -> List[Dict[str, str]]:        """返回格式化的上下文列表。"""        return [{"role": e.role, "content": e.content} for e in self.entries]    def get_summary(self) -> str:        """返回可读的 memory 摘要。"""        lines = []        for e in self.entries:            prefix = {"user": "用户", "assistant": "助手", "tool_call": "工具"}.get(e.role, e.role)            content_preview = e.content[:80] + ("..." if len(e.content) > 80 else "")            lines.append(f"[{prefix}] {content_preview}")        return "\n".join(lines)    @property    def total_tokens(self) -> int:        return sum(e.token_estimate for e in self.entries)    def _truncate_if_needed(self):        """滑窗截断：保留最近的记录，直到总 token 在预算内。"""        while self.total_tokens > self.max_tokens and len(self.entries) > 1:            self.entries.pop(0)    def reset(self):        self.entries.clear()    def __len__(self):        return len(self.entries)

In [ ]:
mem = AgentMemory(max_tokens=200)mem.add_user_message("什么是 LoRA？")mem.add_assistant_message("LoRA 是一种参数高效微调方法，核心思想是低秩分解。")mem.add_user_message("它和 QLoRA 有什么区别？")mem.add_assistant_message("QLoRA 在 LoRA 基础上增加了 NF4 量化和双重量化。")mem.add_user_message("RAG 又是什么？")mem.add_assistant_message("RAG 是检索增强生成，让模型能访问外部知识库。")print(f"Memory 条目数: {len(mem)}")print(f"估算 token 数: {mem.total_tokens}")print(f"\nMemory 内容:")print(mem.get_summary())mem_small = AgentMemory(max_tokens=50)for i in range(10):    mem_small.add_user_message(f"这是第 {i+1} 条很长的消息，用来测试截断功能。")print(f"\n截断测试: 添加了 10 条消息，保留了 {len(mem_small)} 条")print(f"总 token 数: {mem_small.total_tokens} (预算: 50)")assert mem_small.total_tokens <= 50print("✅ Memory 截断功能正常")

## Part 7：带 Memory 的多轮 Agent将 `SimpleReActAgent` 升级为支持多轮对话的 `MultiTurnAgent`：- 每轮对话结束后，把用户问题、Agent 回答和工具调用记录写入 Memory- 下一轮对话时，planner 可以看到 Memory 上下文- 支持指代消解（如"它"、"刚才那个"）

In [ ]:
def multi_turn_planner(    user_query: str,    scratchpad: List[Dict[str, Any]],    tools: List[str],    memory_context: List[Dict[str, str]] = None,) -> Dict[str, Any]:    """    支持多轮上下文的 planner。    在真实系统中，memory_context 会和 user_query、scratchpad 一起    拼入 LLM prompt。这里用规则模拟多轮理解。    """    resolved_query = user_query    if memory_context:        last_topic = None        for entry in reversed(memory_context):            content = entry["content"].lower()            for kw in ["lora", "qlora", "rag", "react", "transformer", "infonce"]:                if kw in content:                    last_topic = kw                    break            if last_topic:                break        if last_topic and any(w in user_query for w in ["它", "这个", "那个", "上面", "刚才"]):            resolved_query = user_query + f" (上文话题: {last_topic})"    return mock_planner(resolved_query, scratchpad, tools)class MultiTurnAgent:    """    支持多轮对话的 Agent。    在 SimpleReActAgent 基础上增加:    - AgentMemory 管理跨轮上下文    - planner 可访问 memory_context    - 每轮结束后自动更新 memory    """    def __init__(        self,        planner: Callable,        registry: ToolRegistry,        max_steps: int = 5,        max_tool_calls: int = 3,        memory_budget: int = 2000,        verbose: bool = True,    ):        self.planner = planner        self.registry = registry        self.max_steps = max_steps        self.max_tool_calls = max_tool_calls        self.memory = AgentMemory(max_tokens=memory_budget)        self.verbose = verbose    def chat(self, user_input: str) -> str:        """处理一轮用户输入，返回 Agent 回复。"""        if self.verbose:            print(f"\n{'='*60}")            print(f"用户: {user_input}")            print(f"Memory 条目: {len(self.memory)}, Token: ~{self.memory.total_tokens}")        self.memory.add_user_message(user_input)        scratchpad: List[Dict[str, Any]] = []        tool_call_count = 0        memory_context = self.memory.get_context()        for step in range(self.max_steps):            decision = self.planner(                user_input, scratchpad, self.registry.list_names(),                memory_context=memory_context,            )            if decision["type"] == "final_answer":                answer = decision["content"]                self.memory.add_assistant_message(answer)                if self.verbose:                    print(f"助手: {answer}")                return answer            if tool_call_count >= self.max_tool_calls:                answer = "工具调用已达上限，基于已有信息作答。"                self.memory.add_assistant_message(answer)                return answer            tool_name = decision["name"]            arguments = decision["arguments"]            if self.verbose:                print(f"  -> {tool_name}({arguments})")            tool_result = safe_execute_tool(self.registry, tool_name, arguments)            tool_call_count += 1            self.memory.add_tool_call(tool_name, arguments, tool_result)            scratchpad.append({                "thought": decision.get("thought", ""),                "action": tool_name,                "arguments": arguments,                "observation": tool_result.to_observation(),                "tool_result_status": tool_result.status,            })        answer = "达到最大步骤数，任务提前终止。"        self.memory.add_assistant_message(answer)        return answer    def show_memory(self):        """显示当前 memory 状态。"""        print(f"\n--- Memory ({len(self.memory)} 条, ~{self.memory.total_tokens} tokens) ---")        print(self.memory.get_summary())    def reset(self):        self.memory.reset()

In [ ]:
multi_agent = MultiTurnAgent(multi_turn_planner, registry, verbose=True)multi_agent.chat("请介绍一下 LoRA")multi_agent.chat("它和 QLoRA 有什么区别？")multi_agent.chat("北京今天天气怎么样？")multi_agent.chat("计算 sqrt(144) + 8")

In [ ]:
multi_agent.show_memory()

## Part 8：完整测试与验证系统级测试，覆盖以下场景：1. Tool 注册与去重2. JSON Schema 参数校验3. 安全执行（成功 / 工具不存在 / 异常）4. Agent 单步工具调用5. Agent 直接回答（无工具）6. Memory 截断7. 多轮 Agent 上下文保持

In [ ]:
def run_test(name, test_fn):    try:        test_fn()        print(f"✅ {name}")        return True    except AssertionError as e:        print(f"❌ {name}: {e}")        return Falseresults = []print("=" * 60)print("Agent 系统测试")print("=" * 60)def test_tool_registry():    r = ToolRegistry()    t = Tool(name="test", description="test", func=lambda: 1, parameters={})    r.register(t)    assert "test" in r.list_names()    assert r.get("test").name == "test"    try:        r.register(t)        assert False, "应该抛出 ValueError"    except ValueError:        passresults.append(run_test("Tool 注册与去重", test_tool_registry))def test_validation():    schema = {        "type": "object",        "properties": {"x": {"type": "integer"}, "y": {"type": "string"}},        "required": ["x"],    }    assert validate_tool_args(schema, {"x": 1})[0] is True    assert validate_tool_args(schema, {"x": 1, "y": "hi"})[0] is True    assert validate_tool_args(schema, {})[0] is False    assert validate_tool_args(schema, {"x": "bad"})[0] is False    assert validate_tool_args(schema, {"x": 1, "z": 2})[0] is Falseresults.append(run_test("JSON Schema 参数校验", test_validation))def test_safe_execute():    r = safe_execute_tool(registry, "calculator", {"expression": "2+3"})    assert r.status == "ok" and r.result == 5    r = safe_execute_tool(registry, "ghost", {})    assert r.status == "error"    r = safe_execute_tool(registry, "calculator", {"expression": "1/0"})    assert r.status == "error"results.append(run_test("安全执行 (成功/失败/异常)", test_safe_execute))def test_agent_single_step():    a = SimpleReActAgent(mock_planner, registry, verbose=False)    r = a.run("计算 2+3")    assert r["tool_calls"] == 1    assert len(r["steps"]) == 1results.append(run_test("Agent 单步工具调用", test_agent_single_step))def test_agent_no_tool():    a = SimpleReActAgent(mock_planner, registry, verbose=False)    r = a.run("你好")    assert r["tool_calls"] == 0results.append(run_test("Agent 直接回答 (无工具)", test_agent_no_tool))def test_memory_truncation():    m = AgentMemory(max_tokens=30)    for i in range(20):        m.add_user_message(f"消息 {i}: 这是一段测试文本。")    assert m.total_tokens <= 30results.append(run_test("Memory 截断", test_memory_truncation))def test_multi_turn():    a = MultiTurnAgent(multi_turn_planner, registry, verbose=False)    a.chat("什么是 LoRA？")    a.chat("北京天气如何？")    assert len(a.memory) >= 4results.append(run_test("多轮 Agent 上下文保持", test_multi_turn))passed = sum(results)total = len(results)print(f"\n{'='*60}")print(f"测试结果: {passed}/{total} 通过")print(f"{'='*60}")

## Part 9：局限分析与面试要点### 9.1 当前系统的局限| 局限 | 原因 | 真实系统的解决方案 ||------|------|--------------------|| 规则化 planner | 未接入 LLM | 替换为 OpenAI Function Calling / 本地模型 || 简单关键词检索 | 无 embedding | Day6 将用向量化检索替代 || 无并行工具调用 | 串行执行 | 支持 parallel function calling || 无长期记忆 | 仅 session 内 | 向量数据库 / KV 存储 || 无安全边界 | calculator 使用 eval | 沙箱执行 / 白名单过滤 |### 9.2 面试高频手写1. **Tool + ToolRegistry**：工具注册与路由机制2. **Agent Loop**：`while step < max: planner -> execute -> observe` 循环3. **safe_execute_tool**：参数校验 + 异常捕获 + 结果规范化4. **AgentMemory**：token 预算管理 + 滑窗截断### 9.3 练习题1. 给 `ToolRegistry` 增加 `unregister` 方法，支持运行时卸载工具。2. 给 `validate_tool_args` 增加 `enum` 类型校验（如 `city` 只能是特定值）。3. 实现一个支持重试的 `safe_execute_tool_with_retry`，最多重试 N 次。4. 把 `mock_planner` 替换成真实 LLM 的 Function Calling 输出。5. 增加一个 `retrieve_docs` 工具，让 Agent 能调用 Day6 的 RAG 检索器。6. 给 `AgentMemory` 增加摘要压缩能力（连接 Day1 的摘要截断策略）。